In [ ]:
import tempfile
from datetime import datetime
from pathlib import Path
from pprint import pprint

import lightgbm as lgb
import matplotlib.pyplot as plt
import mlflow
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv("data/exp02/train.csv")

In [ ]:
mlflow.set_tracking_uri("http://localhost:5001")
mlflow.lightgbm.autolog(disable=False)
experiment = mlflow.get_experiment_by_name("exp02")
run_name = f"01_base_model_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

with mlflow.start_run(experiment_id=experiment.experiment_id, run_name=run_name) as run:
    source_file = globals().get("__vsc_ipynb_file__", "notebook.ipynb")
    mlflow.set_tag("mlflow.source.name", source_file)

    # Split data into features (X) and target (y)
    X = df.drop(["defect"], axis=1)
    y = df["defect"]

    # First split into train+val (80%) and test (20%)
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    # Create dataset for LightGBM
    train_data = lgb.Dataset(X_train, label=y_train)
    val_data = lgb.Dataset(X_val, label=y_val)

    # Set parameters for LightGBM
    params = {
        "objective": "binary",
        "metric": "binary_logloss",
        "boosting_type": "gbdt",
        "num_leaves": 31,
        "learning_rate": 0.05,
        "feature_fraction": 0.9,
    }

    # Train model
    num_round = 100
    lgb_model = lgb.train(
        params, train_data, num_round, valid_sets=[train_data, val_data]
    )

    # Evaluate on validation set
    threshold = 0.5
    val_predictions = (lgb_model.predict(X_val) > threshold).astype(int)
    val_report = classification_report(y_val, val_predictions, output_dict=True)
    print("Validation Set Performance:")
    pprint(val_report)
    mlflow.log_metric("val_precision", val_report["1"]["precision"])
    mlflow.log_metric("val_recall", val_report["1"]["recall"])
    mlflow.log_metric("val_accuracy", val_report["accuracy"])


In [ ]:
with mlflow.start_run(run_id=run.info.run_id) as run:
    # Evaluate on test set
    df_test = pd.read_csv("data/exp02/test.csv")
    X_test = df_test.drop(["defect"], axis=1)
    y_test = df_test["defect"]

    test_predictions = (lgb_model.predict(X_test) > threshold).astype(int)
    test_report = classification_report(y_test, test_predictions, output_dict=True)
    print("Test Set Performance:")
    pprint(test_report)
    mlflow.log_metric("test_precision", test_report["1"]["precision"])
    mlflow.log_metric("test_recall", test_report["1"]["recall"])
    mlflow.log_metric("test_accuracy", test_report["accuracy"])

    disp = ConfusionMatrixDisplay.from_predictions(y_test, test_predictions)
    with tempfile.TemporaryDirectory() as temp_dir:
        path = Path(temp_dir) / "confusion_matrix.png"
        plt.savefig(path)
        mlflow.log_artifact(local_path=path)

    mlflow.lightgbm.log_model(
        lgb_model,
        artifact_path="model",
        signature=mlflow.models.infer_signature(X_test, test_predictions),
        input_example=X_test.iloc[:5],
    )